### Step 1: 	Hello, Data! Load raw CSV, display first 3 rows

1. Chatgpt was used to add new column named "ShippingAddressID" is added to the dataset and the rows are populated with a 5-digit unique numbers to be served as foreign key to another data set containing the shipping details

2. Another column named "CouponCode" is added programatically, and the rows are populated with another 5 digit random keys.

In [195]:
from importlib import reload
import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

import modules.FileManager
reload(modules.FileManager)

import modules.DBManager
reload(modules.DBManager)

# Path to the CSV file containing 
sales_records_path = "data/1000_sales_records_with_shipping_address_id.csv"
address_path = "data/shipping_addresses.csv"
num_of_rows_needed = 500
#Loads the CSV file
fileManager = modules.FileManager.FileManager(sales_records_path, num_of_rows_needed)
sales_data = fileManager.get_data()
address_data = pd.read_csv(address_path)

db_manager = modules.DBManager.DBManager(address_data)
db_manager._drop_and_create_table()
db_manager._insert_into_table()
addresses_from_database = db_manager._fect_all()


# Merge sales_data with address_data on 'Shipping Address ID'
sales_shipping_data = pd.merge(sales_data, addresses_from_database, left_on='Shipping Address ID', right_on='shipping_address_id', how='inner')

# Get the number of rows in the DataFrame
num_rows = len(sales_shipping_data)

# Generate a list of random 5-digit numbers (10000 to 99999)
random_ids = [random.randint(10000, 99999) for _ in range(num_rows)]

# Add the new 'Coupon Code' column to the DataFrame
sales_shipping_data['COUPON CODE'] = random_ids
print("New 'Coupon Code' column added with random 5-digit numbers.")

# Print the first 3 rows of the data
sales_shipping_data.head(3)


✅ Successfully loaded 500 rows from data/1000_sales_records_with_shipping_address_id.csv
🔁 Shipping_Addresses table dropped and recreated.
994 rows inserted!


c:\Users\MOSTAFA\Desktop\ML Programming\Lab 2\modules\DBManager.py:73: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql_query, conn)


✅ Successfully loaded 4970 rows from Shipping_Addresses table.
New 'Coupon Code' column added with random 5-digit numbers.


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,...,Total Profit,Shipping Address ID,shipping_address_id,country,street,city,state/province,postal_code,phone_number,COUPON CODE
0,Middle East and North Africa,Libya,Cosmetics,Offline,M,10/18/2014,686800706,10/31/2014,8446,437.2,...,1468506.02,84026,84026,Libya,1069 Main St,Sydney,ENG,33769,Phone Number,80265
1,Middle East and North Africa,Libya,Cosmetics,Offline,M,10/18/2014,686800706,10/31/2014,8446,437.2,...,1468506.02,84026,84026,Libya,1069 Main St,Sydney,ENG,33769,Phone Number,81603
2,Middle East and North Africa,Libya,Cosmetics,Offline,M,10/18/2014,686800706,10/31/2014,8446,437.2,...,1468506.02,84026,84026,Libya,1069 Main St,Sydney,ENG,33769,Phone Number,70049


### Step 2: Pick the Right Container. Justify dict vs namedtuple vs sets(1–2 sentences)

1. Dictionaries (dict): Best for mapping unique keys to values, offering fast lookups and mutable data structures for dynamic key-value pairs. 

2. Named Tuples (namedtuple): Ideal for creating lightweight, immutable objects with readable attribute names, serving as a clean and efficient alternative to a dictionary for representing a fixed set of data. 

3. Sets (set): Designed for storing unique, unordered elements, providing very fast membership testing and operations like union and intersection. 

#### In this case, Dictionary would be a better choice than namedtuple or sets

### Step 3: Implement Functions and  Data structure and use them to populate a data structure (Dictionary)


### Step 4: Bulk Loaded - Map data structures from dataframes to dictionaries

In [196]:
# Convert data into a dictionary
records_dict = sales_shipping_data.to_dict(orient="records")
print(records_dict[:3])  # Print the first 3 records to verify



[{'Region': 'Middle East and North Africa', 'Country': 'Libya', 'Item Type': 'Cosmetics', 'Sales Channel': 'Offline', 'Order Priority': 'M', 'Order Date': '10/18/2014', 'Order ID': 686800706, 'Ship Date': '10/31/2014', 'Units Sold': 8446, 'Unit Price': 437.2, 'Unit Cost': 263.33, 'Total Revenue': 3692591.2, 'Total Cost': 2224085.18, 'Total Profit': 1468506.02, 'Shipping Address ID': 84026, 'shipping_address_id': 84026, 'country': 'Libya', 'street': '1069 Main St', 'city': 'Sydney', 'state/province': 'ENG', 'postal_code': 33769, 'phone_number': 'Phone Number', 'COUPON CODE': 80265}, {'Region': 'Middle East and North Africa', 'Country': 'Libya', 'Item Type': 'Cosmetics', 'Sales Channel': 'Offline', 'Order Priority': 'M', 'Order Date': '10/18/2014', 'Order ID': 686800706, 'Ship Date': '10/31/2014', 'Units Sold': 8446, 'Unit Price': 437.2, 'Unit Cost': 263.33, 'Total Revenue': 3692591.2, 'Total Cost': 2224085.18, 'Total Profit': 1468506.02, 'Shipping Address ID': 84026, 'shipping_address_i

### Step 5: Quick Profiling - Min/mean/max price, unique city count (set)

In [197]:
sales_shipping_data.info()

numerical_columns = sales_shipping_data.select_dtypes(include=np.float64)

# Extract the list of total profits from the dictionary
total_profits = sales_shipping_data['Total Profit']

print(f"Total number of columns: {len(sales_shipping_data.columns)}")
print(f"Total number of rows: {len(sales_shipping_data)}")

# Calculate statistics for and print the result
for c in sales_shipping_data.columns:
    print(f"\n--- Column: '{c}' ---")

    # Get the count of unique values
    unique_count = sales_shipping_data[c].nunique()
    print(f"Unique Values: {unique_count}")

    # Get the number of missing values
    missing_count = sales_shipping_data[c].isnull().sum()
    print(f"Missing Values: {missing_count}")

    
    if(c in numerical_columns.columns):
        print(f"Min: ${numerical_columns[c].min():,.2f}")
        print(f"Max: ${numerical_columns[c].max():,.2f}")
        print(f"Mean: ${numerical_columns[c].mean():,.2f}")


# Extract the list of countries
countries = sales_shipping_data['Country']

# Use a set to get the unique countries and count them
unique_countries_count = len(set(countries)) 
print(f"\nTotal number of unique countries: {unique_countries_count}")

# numerical_columns.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Region               2500 non-null   object 
 1   Country              2500 non-null   object 
 2   Item Type            2500 non-null   object 
 3   Sales Channel        2500 non-null   object 
 4   Order Priority       2500 non-null   object 
 5   Order Date           2500 non-null   object 
 6   Order ID             2500 non-null   int64  
 7   Ship Date            2500 non-null   object 
 8   Units Sold           2500 non-null   int64  
 9   Unit Price           2500 non-null   float64
 10  Unit Cost            2500 non-null   float64
 11  Total Revenue        2500 non-null   float64
 12  Total Cost           2500 non-null   float64
 13  Total Profit         2500 non-null   float64
 14  Shipping Address ID  2500 non-null   int64  
 15  shipping_address_id  2500 non-null   i

#### Step 6: Spot the Grime - Identify at least three dirty data cases

In [198]:
# Data Cleaning and Normalization

# 1. Remove leading/trailing whitespace from string columns
for col in sales_shipping_data.select_dtypes(include='object').columns:
  sales_shipping_data[col] = sales_shipping_data[col].str.strip()


# 2. Remove the columns 'country' and 'shipping_address_id' since they are dupplicated and they come from both datasets
sales_shipping_data.drop(columns=['country'], inplace=True)
sales_shipping_data.drop(columns=['shipping_address_id'], inplace=True)
    
# 3. Standardize column names (lowercase, replace spaces with underscores)
sales_shipping_data.columns = [col.strip().lower().replace(' ', '_') for col in sales_shipping_data.columns]

# 4. Columns should be numeric whenever possible - replacing online/offline with 1/0
for i in range(len(sales_shipping_data)):
  if sales_shipping_data.loc[i, "sales_channel"].lower() == "online":
    sales_shipping_data.loc[i, "sales_channel"] = 1
  elif sales_shipping_data.loc[i, "sales_channel"].lower() == "offline":
    sales_shipping_data.loc[i, "sales_channel"] = 0
  else:
    sales_shipping_data.loc[i, "sales_channel"] = np.nan

# Check result"
sales_shipping_data.head(len(sales_shipping_data))


,region,country,item_type,sales_channel,order_priority,order_date,order_id,ship_date,units_sold,unit_price,...,total_revenue,total_cost,total_profit,shipping_address_id,street,city,state/province,postal_code,phone_number,coupon_code
0,Middle East and North Africa,Libya,Cosmetics,0,M,10/18/2014,686800706,10/31/2014,8446,437.20,...,3692591.20,2224085.18,1468506.02,84026,1069 Main St,Sydney,ENG,33769,Phone Number,80265
1,Middle East and North Africa,Libya,Cosmetics,0,M,10/18/2014,686800706,10/31/2014,8446,437.20,...,3692591.20,2224085.18,1468506.02,84026,1069 Main St,Sydney,ENG,33769,Phone Number,81603
2,Middle East and North Africa,Libya,Cosmetics,0,M,10/18/2014,686800706,10/31/2014,8446,437.20,...,3692591.20,2224085.18,1468506.02,84026,1069 Main St,Sydney,ENG,33769,Phone Number,70049
3,Middle East and North Africa,Libya,Cosmetics,0,M,10/18/2014,686800706,10/31/2014,8446,437.20,...,3692591.20,2224085.18,1468506.02,84026,1069 Main St,Sydney,ENG,33769,Phone Number,52285
4,Middle East and North Africa,Libya,Cosmetics,0,M,10/18/2014,686800706,10/31/2014,8446,437.20,...,3692591.20,2224085.18,1468506.02,84026,1069 Main St,Sydney,ENG,33769,Phone Number,22189
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,Middle East and North Africa,Jordan,Fruits,1,C,5/15/2017,521787345,6/25/2017,7325,9.33,...,68342.25,50689.00,17653.25,75437,4233 Main St,Toronto,ENG,67417,Phone Number,86658
2496,Middle East and North Africa,Jordan,Fruits,1,C,5/15/2017,521787345,6/25/2017,7325,9.33,...,68342.25,50689.00,17653.25,75437,4233 Main St,Toronto,ENG,67417,Phone Number,18794
2497,Middle East and North Africa,Jordan,Fruits,1,C,5/15/2017,521787345,6/25/2017,7325,9.33,...,68342.25,50689.00,17653.25,75437,4233 Main St,Toronto,ENG,67417,Phone Number,17380
2498,Middle East and North Africa,Jordan,Fruits,1,C,5/15/2017,521787345,6/25/2017,7325,9.33,...,68342.25,50689.00,17653.25,75437,4233 Main St,Toronto,ENG,67417,Phone Number,64467
